# Normalizar los datos

Importar las librerías


*   Pandas: se utiliza para manipulación y análisis de datos
*   NumPy: librería de Python se utiliza para operaciones matemáticas, arrays, etc.

*as pd* es el alias de la librería para no escribir más "código"



*StandardScaler* y *MinMaxScaler*
Son clases (plantillas) que crean objetos capaces de escalar/normalizar datos.

*SimpleImputer* rellena los valores vacíos (nulos) automáticamente

In [51]:
!pip install pandas numpy scikit-learn -q

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

Cargar el data set

*df.shape* te muestra las filas y columnas

In [52]:
from google.colab import files
uploaded = files.upload()  # subí tu sales_data_sample.csv

df = pd.read_csv('sales_data_sample.csv', encoding='latin1')
print("Shape original:", df.shape)

Saving sales_data_sample.csv to sales_data_sample (2).csv
Shape original: (2823, 25)


Eliminar columnas innecesarias

In [53]:
cols_drop = [
    'ORDERNUMBER', 'ORDERLINENUMBER', 'ORDERDATE',
    'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE',
    'ADDRESSLINE1', 'ADDRESSLINE2', 'POSTALCODE',
    'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'STATE'
]
df = df.drop(columns=cols_drop)

Imputar nulos

In [54]:
cols_cat = ['STATUS', 'PRODUCTLINE', 'COUNTRY', 'TERRITORY', 'DEALSIZE', 'CITY']
imputer = SimpleImputer(strategy='most_frequent')
df[cols_cat] = imputer.fit_transform(df[cols_cat])

Normalizar columnas numéricas

In [55]:
cols_num = ['QUANTITYORDERED', 'PRICEEACH', 'SALES', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'MSRP']
scaler = MinMaxScaler()
df[cols_num] = scaler.fit_transform(df[cols_num])

Guardar CSV limpio

In [56]:
df.to_csv('sales_clean.csv', index=False)
print("✅ Dataset limpio guardado. Shape:", df.shape)
print(df.head())

files.download('sales_clean.csv')

✅ Dataset limpio guardado. Shape: (2823, 13)
   QUANTITYORDERED  PRICEEACH     SALES   STATUS    QTR_ID  MONTH_ID  YEAR_ID  \
0         0.263736   0.941193  0.175644  Shipped  0.000000  0.090909      0.0   
1         0.307692   0.744940  0.167916  Shipped  0.333333  0.363636      0.0   
2         0.384615   0.928063  0.250150  Shipped  0.666667  0.545455      0.0   
3         0.428571   0.771061  0.240030  Shipped  0.666667  0.636364      0.0   
4         0.472527   1.000000  0.347273  Shipped  1.000000  0.818182      0.0   

   PRODUCTLINE      MSRP           CITY COUNTRY TERRITORY DEALSIZE  
0  Motorcycles  0.342541            NYC     USA      EMEA    Small  
1  Motorcycles  0.342541          Reims  France      EMEA    Small  
2  Motorcycles  0.342541          Paris  France      EMEA   Medium  
3  Motorcycles  0.342541       Pasadena     USA      EMEA   Medium  
4  Motorcycles  0.342541  San Francisco     USA      EMEA   Medium  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# RAG + Chatbot

Instalar las dependencias/librerías

API 🗝

In [57]:
!pip install langchain langchain-community langchain-mistralai \ chromadb sentence-transformers mistralai -q

In [58]:
import os
import pandas as pd
from google.colab import userdata
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')
print("✅ Importaciones listas")

✅ Importaciones listas


Cargar el CSV limpio

In [59]:
# Cargar el CSV limpio
from google.colab import files
uploaded = files.upload()  # ← te va a abrir un botón para subir el archivo
df = pd.read_csv('sales_clean.csv')
print("✅ Cargado correctamente. Shape:", df.shape)

Saving sales_clean.csv to sales_clean (1).csv
✅ Cargado correctamente. Shape: (2823, 13)


RAG necesita texto, no números — cada fila se convierte en una oración

*f"..."* permite meter variables dentro del texto con *{}*

Convertimos en texto por que *ChromaDB* y los *embedding* trabajan con texto, no con tablas numéricas

In [60]:
def fila_a_texto(row):
    return (
        f"Venta: {row['PRODUCTLINE']} | País: {row['COUNTRY']} | "
        f"Ciudad: {row['CITY']} | Territorio: {row['TERRITORY']} | "
        f"Cantidad ordenada (norm): {row['QUANTITYORDERED']:.2f} | "
        f"Precio unitario (norm): {row['PRICEEACH']:.2f} | "
        f"Total venta (norm): {row['SALES']:.2f} | "
        f"MSRP (norm): {row['MSRP']:.2f} | "
        f"Tamaño del deal: {row['DEALSIZE']} | "
        f"Estado: {row['STATUS']} | "
        f"Trimestre: {row['QTR_ID']:.0f} | Mes: {row['MONTH_ID']:.0f} | Año: {row['YEAR_ID']:.0f}"
    )

documentos = [
    Document(page_content=fila_a_texto(row), metadata={"index": i})
    for i, row in df.iterrows()
]
print(f"✅ {len(documentos)} documentos creados")

✅ 2823 documentos creados


Embedir

In [61]:
print("⏳ Creando embeddings (puede tardar 2-3 minutos)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    # Modelo multilingüe: entiende preguntas en español e inglés
)

vectorstore = Chroma.from_documents(documentos, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
print("✅ Vector store listo")

⏳ Creando embeddings (puede tardar 2-3 minutos)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store listo


Configurar Mistral

*ChatMistralAI* crea la conexión con la API de MIstral usando la key que guardé en "Secrets"

In [62]:
llm = ChatMistralAI(model="mistral-small-latest", temperature=0.1)

prompt_template = """
Eres un analista de ventas experto. Usa SOLO la información del contexto para responder.
Si no podés responder con los datos del contexto, decilo claramente.
Responde siempre en español.

Contexto:
{context}

Pregunta: {question}

Respuesta:
"""
prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

def formatear_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | formatear_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("✅ Cadena RAG lista")

✅ Cadena RAG lista


Chatbot

In [64]:
print("\n🤖 Chatbot listo. Escribí 'salir' para terminar.\n")

while True:
    pregunta = input("Vos: ")
    if pregunta.lower() in ['salir', 'exit', 'quit']:
        print("¡Hasta luego!")
        break
    if not pregunta.strip():
        continue
    respuesta = qa_chain.invoke(pregunta)
    print(f"\n🤖 Bot: {respuesta}\n")


🤖 Chatbot listo. Escribí 'salir' para terminar.

Vos: en que pais se hizo mas ventas?

🤖 Bot: Según el contexto proporcionado, todas las ventas registradas son del **Reino Unido (UK)**. No hay datos que indiquen ventas en otros países.

**Respuesta:** Las ventas se realizaron en el **Reino Unido (UK)**.

Vos: salir
¡Hasta luego!


Copia en git hub

In [65]:
!git clone https://github.com/miliacoss0/Boot-con-Mistral.ia.git

fatal: destination path 'Boot-con-Mistral.ia' already exists and is not an empty directory.
